# 03 — DML no Delta Lake (bronze)

Demonstra as 3 operacoes de DML suportadas pelo Delta Lake:
- **INSERT** via `merge` (upsert)
- **UPDATE** condicional
- **DELETE** condicional

In [1]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from delta.tables import DeltaTable

load_dotenv()

MINIO_ENDPOINT   = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
BRONZE           = os.getenv("MINIO_BRONZE_BUCKET")

spark = (
    SparkSession.builder
    .appName("dml-delta-bronze")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark versao:", spark.version)

Spark versao: 3.5.0


## INSERT — novos registros via merge

Insere novos clientes e apolices sem duplicar registros existentes.

In [2]:
dest_cliente = f"s3a://{BRONZE}/cliente"
dt_cliente   = DeltaTable.forPath(spark, dest_cliente)

print("cliente antes do INSERT:", spark.read.format("delta").load(dest_cliente).count(), "registros")

novos_clientes = spark.createDataFrame([
    (11, "Patricia Nunes",   "111.222.333-44", "1993-07-20", "patricia@email.com", 1),
    (12, "Rafael Oliveira",  "555.666.777-88", "1987-02-14", "rafael@email.com",   2),
], ["id_cliente", "nome", "cpf", "data_nascimento", "email", "id_endereco"])

(
    dt_cliente.alias("t")
    .merge(
        novos_clientes.alias("s"),
        "t.id_cliente = s.id_cliente"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("cliente apos  INSERT:", spark.read.format("delta").load(dest_cliente).count(), "registros")

cliente antes do INSERT: 10 registros
cliente apos  INSERT: 12 registros


In [3]:
dest_sinistro = f"s3a://{BRONZE}/sinistro"
dt_sinistro   = DeltaTable.forPath(spark, dest_sinistro)

print("sinistro antes do INSERT:", spark.read.format("delta").load(dest_sinistro).count(), "registros")

novos_sinistros = spark.createDataFrame([
    (9,  "2024-09-03", "Colisão com animal na pista",  4200.00, "Em análise", 7),
    (10, "2024-10-15", "Granizo danificou lataria",    2800.00, "Em análise", 10),
], ["id_sinistro", "data_ocorrencia", "descricao", "valor_prejuizo", "status", "id_apolice"])

(
    dt_sinistro.alias("t")
    .merge(
        novos_sinistros.alias("s"),
        "t.id_sinistro = s.id_sinistro"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("sinistro apos  INSERT:", spark.read.format("delta").load(dest_sinistro).count(), "registros")

sinistro antes do INSERT: 8 registros
sinistro apos  INSERT: 10 registros


## UPDATE — atualizacao condicional

In [4]:
dest_apolice = f"s3a://{BRONZE}/apolice"
dt_apolice   = DeltaTable.forPath(spark, dest_apolice)

print("apolice ANTES do UPDATE (cobertura Basica):")
spark.read.format("delta").load(dest_apolice).filter(col("cobertura") == "Básica").show(truncate=False)

# Reajuste de 15% no prêmio das apólices básicas
dt_apolice.update(
    condition=col("cobertura") == "Básica",
    set={"valor_premio": col("valor_premio") * lit(1.15)}
)

print("apolice APOS UPDATE (reajuste 15% nas Basicas):")
spark.read.format("delta").load(dest_apolice).filter(col("cobertura") == "Básica").show(truncate=False)

apolice ANTES do UPDATE (cobertura Basica):
+----------+--------+-----------+----------+------------+---------+----------+--------+
|id_apolice|numero  |data_inicio|data_fim  |valor_premio|cobertura|id_cliente|id_carro|
+----------+--------+-----------+----------+------------+---------+----------+--------+
|2         |AP-00002|2024-02-01 |2025-02-01|1200.0      |Básica   |2         |2       |
|4         |AP-00004|2024-03-01 |2025-03-01|900.0       |Básica   |4         |4       |
|7         |AP-00007|2024-01-20 |2025-01-20|800.0       |Básica   |7         |7       |
|10        |AP-00010|2024-02-20 |2025-02-20|1100.0      |Básica   |10        |10      |
+----------+--------+-----------+----------+------------+---------+----------+--------+

apolice APOS UPDATE (reajuste 15% nas Basicas):
+----------+--------+-----------+----------+-----------------+---------+----------+--------+
|id_apolice|numero  |data_inicio|data_fim  |valor_premio     |cobertura|id_cliente|id_carro|
+----------+-----

In [5]:
# Atualiza status de sinistros em analise para Aprovado
print("sinistro ANTES do UPDATE:")
spark.read.format("delta").load(dest_sinistro).show(truncate=False)

dt_sinistro.update(
    condition=col("status") == "Em análise",
    set={"status": lit("Aprovado")}
)

print("sinistro APOS UPDATE:")
spark.read.format("delta").load(dest_sinistro).show(truncate=False)

sinistro ANTES do UPDATE:
+-----------+---------------+---------------------------+--------------+----------+----------+
|id_sinistro|data_ocorrencia|descricao                  |valor_prejuizo|status    |id_apolice|
+-----------+---------------+---------------------------+--------------+----------+----------+
|1          |2024-03-10     |Colisão frontal em avenida |15000.0       |Pago      |1         |
|2          |2024-04-22     |Roubo do veículo           |45000.0       |Em análise|2         |
|3          |2024-02-05     |Alagamento por chuva       |8000.0        |Pago      |3         |
|4          |2024-06-15     |Colisão traseira           |5500.0        |Recusado  |4         |
|5          |2024-05-30     |Incêndio no veículo        |32000.0       |Em análise|5         |
|6          |2024-07-08     |Quebra de vidro            |1200.0        |Pago      |6         |
|7          |2024-03-25     |Colisão em estacionamento  |3800.0        |Pago      |8         |
|8          |2024-08-11 

## DELETE — exclusao condicional

In [6]:
print(f"sinistro ANTES do DELETE: {spark.read.format('delta').load(dest_sinistro).count()} registros")

# Remove sinistros com status Recusado
dt_sinistro.delete(condition=col("status") == "Recusado")

print(f"sinistro APOS  DELETE: {spark.read.format('delta').load(dest_sinistro).count()} registros")
spark.read.format("delta").load(dest_sinistro).show(truncate=False)

sinistro ANTES do DELETE: 10 registros
sinistro APOS  DELETE: 9 registros
+-----------+---------------+---------------------------+--------------+--------+----------+
|id_sinistro|data_ocorrencia|descricao                  |valor_prejuizo|status  |id_apolice|
+-----------+---------------+---------------------------+--------------+--------+----------+
|1          |2024-03-10     |Colisão frontal em avenida |15000.0       |Pago    |1         |
|2          |2024-04-22     |Roubo do veículo           |45000.0       |Aprovado|2         |
|3          |2024-02-05     |Alagamento por chuva       |8000.0        |Pago    |3         |
|5          |2024-05-30     |Incêndio no veículo        |32000.0       |Aprovado|5         |
|6          |2024-07-08     |Quebra de vidro            |1200.0        |Pago    |6         |
|7          |2024-03-25     |Colisão em estacionamento  |3800.0        |Pago    |8         |
|8          |2024-08-11     |Roubo de acessórios        |2500.0        |Aprovado|9       

In [7]:
dest_carro = f"s3a://{BRONZE}/carro"
dt_carro   = DeltaTable.forPath(spark, dest_carro)

print(f"carro ANTES do DELETE: {spark.read.format('delta').load(dest_carro).count()} registros")

# Remove carros fabricados antes de 2018
dt_carro.delete(condition=col("ano") < lit(2018))

print(f"carro APOS  DELETE: {spark.read.format('delta').load(dest_carro).count()} registros")

carro ANTES do DELETE: 10 registros
carro APOS  DELETE: 9 registros


## Historico de versoes (Time Travel)

In [8]:
for table in ["cliente", "apolice", "sinistro", "carro"]:
    dest    = f"s3a://{BRONZE}/{table}"
    dt      = DeltaTable.forPath(spark, dest)
    history = dt.history().select("version", "timestamp", "operation")
    print(f"{table}:")
    history.show(truncate=False)

cliente:
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|1      |2026-05-06 23:27:07|MERGE    |
|0      |2026-05-06 23:25:10|WRITE    |
+-------+-------------------+---------+

apolice:
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|1      |2026-05-06 23:27:46|UPDATE   |
|0      |2026-05-06 23:25:04|WRITE    |
+-------+-------------------+---------+

sinistro:
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|3      |2026-05-06 23:28:24|DELETE   |
|2      |2026-05-06 23:28:08|UPDATE   |
|1      |2026-05-06 23:27:29|MERGE    |
|0      |2026-05-06 23:25:21|WRITE    |
+-------+-------------------+---------+

carro:
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|1      |2026-05-06 23:28:39|DELETE   |
|0